# Funnel Analysis to Calculate Step-by-step Drop-off Rates

In [ ]:
# Page navigation logs by user（dummy data）
# Contains data where the same user has viewed the same page multiple times (duplicate data)
funnel_logs = [
    {"user_id": "U001", "step": "top_page"},
    {"user_id": "U001", "step": "product_page"},
    {"user_id": "U001", "step": "cart"},
    {"user_id": "U001", "step": "purchase"},
    {"user_id": "U002", "step": "top_page"},
    {"user_id": "U002", "step": "product_page"},
    {"user_id": "U002", "step": "top_page"}, # Duplicate access from U002
    {"user_id": "U003", "step": "top_page"},
    {"user_id": "U004", "step": "top_page"},
    {"user_id": "U004", "step": "product_page"},
    {"user_id": "U004", "step": "cart"},
    {"user_id": "U005", "step": "top_page"},
    {"user_id": "U005", "step": "product_page"},
    {"user_id": "U005", "step": "product_page"}, # Duplicate access from U005
]

# The correct order of the funnel
FUNNEL_STEPS = ["top_page", "product_page", "cart", "purchase"]

def get_users_at_step(logs, step):
    """
    Return a set of unique IDs who reached the given funnel step.

    Args:
        logs (list): Funnel logs. Each log is a dictionary containing user_id and step.
        step (str): The name of the funnel step to filter by.

    Returns:
        set: A set of unique user IDs who reached the given step.
    """
    users = set()
    for log in logs:
        if log["step"] == step:
            users.add(log["user_id"])

    return users

def analyze_funnel(logs, steps):
    """
    Calculate the number of users, transition_rate, and drop-off rate for each funnel step.

    Args:
        logs (list): Funnel logs. Each log is a dictionary containing user_id and step.
        steps (list): Ordered list of funnel step names.

    Returns:
        list: A list of dictionaries for each step containing:
            - step (str): Step name.
            - count (int): Number of unique users at this step.
            - transition_rate (float or None): Percentage of users who moved to the next step. None for the last step.
            - drop_rate (float or None): Percentage of users who dropped off. None for the last step.
    """
    result = []
    for i, step in enumerate(steps):
        count = len(get_users_at_step(logs, step))

        if i < len(steps) - 1:
            next_count = len(get_users_at_step(logs, steps[i + 1]))
            transition_rate = (next_count / count) * 100 if count > 0 else 0
            drop_rate = 100 - transition_rate
        else:
            transition_rate = None
            drop_rate = None
        
        result.append({
            "step": step,
            "count": count,
            "transition_rate": transition_rate,
            "drop_rate": drop_rate
        })

    return result

def print_funnel_summary(analyzed_data):
    """
    Print the number of users, transition rate, and drop-off rate for each funnel step.

    Args:
        analyzed_data (list): A list of dictionaries for each step containing:
            - step (str): Step name.
            - count (int) : Number of unique users at this step.
            - transition_rate (float or None): Percentage of users who moved to the next step. None for the last step.
            - drop_rate (float or None): Percentage of users who dropped off. None for the last step.
    """
    
    for data in analyzed_data:
        step = data["step"]
        count = data["count"]
        t_rate = data["transition_rate"]
        d_rate = data["drop_rate"]

        if t_rate is None:
            print(f"{step:<15}: {count:>2} users (last step)")
        else:
            print(f"{step:<15}: {count:>2} users → transition: {t_rate:5.1f}% drop-off: {d_rate:5.1f}%")


funnel_results = analyze_funnel(funnel_logs, FUNNEL_STEPS)
print_funnel_summary(funnel_results)

top_page       :  5 users → transition:  80.0% drop-off:  20.0%
product_page   :  4 users → transition:  50.0% drop-off:  50.0%
cart           :  2 users → transition:  50.0% drop-off:  50.0%
purchase       :  1 users (last step)


## What I Learned

- Used `set()` to remove duplicate user access logs.
- Created a function to get unique users for each funnel step.
- Calculated transition rates and drop-off rates between funnel steps.
- Learned that funnel analysis should compare overlapping users between steps, not only total counts.

## Next Step

- Improve the transition calculation by using set intersection (`current_users & next_users`).